In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg, year,month ,max

spark = SparkSession.builder.appName("CaseStudy1").getOrCreate()

In [3]:
customers = spark.read.csv("./data/customers.csv",
                           header=True,
                           inferSchema=True)

products = spark.read.csv("./data/products.csv",
                          header=True,
                          inferSchema=True)

orders = spark.read.csv("./data/orders.csv",
                        header=True,
                        inferSchema=True)

order_items = spark.read.csv("./data/order_items.csv",
                             header=True,
                             inferSchema=True)

returns = spark.read.csv("./data/returns.csv",
                         header=True,
                         inferSchema=True)

In [4]:
customers.printSchema()
products.printSchema()
orders.printSchema()
order_items.printSchema()
returns.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_cost: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- selling_price: double (nullable = true)

root
 |-- return_id: integer (nullable = true)
 |-- order_id: integer (nullable = tru

In [5]:
customers.show(5)
products.show(5)
orders.show(5)
order_items.show(5)
returns.show(5)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electroni

In [6]:
# QUES 1 :- Total number of customers, products, orders, and returned orders
total_customers = customers.count()
total_products = products.count()
total_orders = orders.count()
total_returns = returns.select("order_id").distinct().count()

print("Total Customers:", total_customers)
print("Total Products:", total_products)
print("Total Orders:", total_orders)
print("Total Returned Orders:", total_returns)

Total Customers: 100000
Total Products: 50000
Total Orders: 1000000
Total Returned Orders: 100000


In [7]:
# QUES 2 :- Total sales amount generated by each product category
sales_by_category = (
    order_items
    .join(products,"product_id")
    .groupBy("category")
    .agg(
        sum(
            col("quantity")*col("selling_price")
        ).alias("total_sales")
    )
)

sales_by_category.show()

[Stage 31:====>                                                   (1 + 11) / 12]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732799999E8|
|        Sports|7.433388681300002E8|
|   Electronics|7.442665041099973E8|
|      Clothing|7.419227945700002E8|
|         Books|7.464907783500013E8|
|        Beauty|7.626693058999996E8|
|          Toys|7.446190723000015E8|
+--------------+-------------------+



In [9]:
# QUES 3 :- Top 10 customers based on total purchase amount
top10 = (
    orders.join(order_items, "order_id")
          .join(customers, "customer_id")
          .groupBy("customer_id", "customer_name")
          .agg(
              sum(order_items.quantity * order_items.selling_price)
              .alias("total_purchase")
          )
          .orderBy(desc("total_purchase"))
          .limit(10)
)

top10.show()

[Stage 49:====>                                                   (1 + 12) / 13]

+-----------+--------------+------------------+
|customer_id| customer_name|    total_purchase|
+-----------+--------------+------------------+
|      93094|Customer_93094|         181569.68|
|      64560|Customer_64560|          169060.4|
|      23289|Customer_23289|161573.79999999996|
|      52275|Customer_52275|153364.78999999995|
|      61218|Customer_61218|         153067.55|
|      52034|Customer_52034|152680.04999999996|
|      40442|Customer_40442|151037.31999999998|
|      60528|Customer_60528|         148691.95|
|      84830|Customer_84830|         148363.84|
|      82593|Customer_82593|         148281.04|
+-----------+--------------+------------------+



In [14]:
# QUES 4 :- Monthly sales trends for the last available year
latest_year = (
    orders.select(max(year("order_date")))
          .collect()[0][0]
)

monthly_sales = (
    orders.join(order_items, "order_id")
          .filter(year("order_date") == latest_year)
          .withColumn("sales", col("quantity") * col("selling_price"))
          .groupBy(month("order_date").alias("month"))
          .agg(sum("sales").alias("total_sales"))
          .orderBy("month")
)

monthly_sales.show()

[Stage 61:=====================>                                   (5 + 8) / 13]

+-----+--------------------+
|month|         total_sales|
+-----+--------------------+
|    1|4.4457777575999975E8|
|    2|4.1536614420000035E8|
|    3|4.4362824540999967E8|
|    4| 4.278209743400001E8|
|    5| 4.448106189500004E8|
|    6|4.3170515405999935E8|
|    7|4.4367051911999947E8|
|    8| 4.410951770199989E8|
|    9| 4.310715260799998E8|
|   10|4.4136378930999917E8|
|   11| 4.336233640399997E8|
|   12|4.4271290834999955E8|
+-----+--------------------+

